# Build dimension table

- `metadata` - just track which release/date we're using
- `dim_cell_lines` - master list of every cell line
- `dim_genes` - master list of genes

In [118]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / 'data'
PROCESSED = DATA / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

## metadata table


In [119]:
from datetime import date

metadata = pd.DataFrame([{
    'schema_version': '1.0',
    'depmap_release': '25Q2',
    'harmonized_date': str(date.today()),
    'pipeline_version': 'v1_initial',
    'primary_key': 'ach_id',
    'secondary_key': 'cvcl_id',
}])

metadata

,schema_version,depmap_release,harmonized_date,pipeline_version,primary_key,secondary_key
0,1.0,25Q2,2026-06-20,v1_initial,ach_id,cvcl_id


In [120]:
metadata.to_parquet(PROCESSED / 'metadata.parquet', index=False)
print('saved metadata')

saved metadata


## dim_cell_lines

Start with File 9 as the base. It has 1840 cell lines with all the metadata I need (lineage, disease, growth pattern etc).

Then add:
- orphan ACHs from fact files (cell lines that exist in fact data but not in File 9)
- HPA-only cell lines (have CVCL but no ACH)

In [121]:
samples = pd.read_csv(DATA / 'nomenclature/9_DepMap_sample_info.csv')
samples.head()

,DepMap_ID,cell_line_name,stripped_cell_line_name,CCLE_Name,alias,COSMICID,sex,source,RRID,WTSI_Master_Cell_ID,...,lineage_sub_subtype,lineage_molecular_subtype,default_growth_pattern,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,Cellosaurus_NCIt_disease,Cellosaurus_NCIt_id,Cellosaurus_issues
0,ACH-000016,SLR 21,SLR21,SLR21_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_V607,NaN,...,NaN,NaN,NaN,NaN,NaN,PT-JnARLB,NaN,Clear cell renal cell carcinoma,C4033,NaN
1,ACH-000032,MHH-CALL-3,MHHCALL3,MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Female,DSMZ,CVCL_0089,NaN,...,b_cell,NaN,NaN,NaN,NaN,PT-p2KOyI,NaN,Childhood B acute lymphoblastic leukemia,C9140,NaN
2,ACH-000033,NCI-H1819,NCIH1819,NCIH1819_LUNG,NaN,NaN,Female,Academic lab,CVCL_1497,NaN,...,NSCLC_adenocarcinoma,NaN,NaN,NaN,NaN,PT-9p1WQv,NaN,Lung adenocarcinoma,C3512,NaN
3,ACH-000043,Hs 895.T,HS895T,HS895T_FIBROBLAST,NaN,NaN,Female,ATCC,CVCL_0993,NaN,...,NaN,NaN,2D: adherent,NaN,NaN,PT-rTUVZQ,NaN,Melanoma,C3224,NaN
4,ACH-000049,HEK TE,HEKTE,HEKTE_KIDNEY,NaN,NaN,NaN,Academic lab,CVCL_WS59,NaN,...,NaN,NaN,NaN,immortalized,NaN,PT-qWYYgr,NaN,NaN,NaN,No information is available about this cell li...


rename to schema names

In [122]:
samples_renamed = samples.drop(columns=['cell_line_name']).rename(columns={
    'DepMap_ID': 'ach_id',
    'stripped_cell_line_name': 'cell_line_name',
    'CCLE_Name': 'ccle_name',
    'RRID': 'cvcl_id',
    'default_growth_pattern': 'growth_pattern',
})

In [123]:
keep_cols = [
    'ach_id', 'cvcl_id', 'cell_line_name', 'ccle_name',
    'primary_disease', 'Subtype', 'lineage', 'lineage_subtype',
    'sex', 'age', 'primary_or_metastasis', 'sample_collection_site',
    'growth_pattern', 'COSMICID', 'Sanger_Model_ID', 'source',
]

dim = samples_renamed[keep_cols].copy()
dim['metadata_source'] = 'file_9'
dim.head()

,ach_id,cvcl_id,cell_line_name,ccle_name,primary_disease,Subtype,lineage,lineage_subtype,sex,age,primary_or_metastasis,sample_collection_site,growth_pattern,COSMICID,Sanger_Model_ID,source,metadata_source
0,ACH-000016,CVCL_V607,SLR21,SLR21_KIDNEY,Kidney Cancer,Renal Cell Carcinoma,kidney,renal_cell_carcinoma,NaN,NaN,Metastasis,kidney,NaN,NaN,NaN,Academic lab,file_9
1,ACH-000032,CVCL_0089,MHHCALL3,MHHCALL3_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Leukemia,"Acute Lymphoblastic Leukemia (ALL), B-cell",blood,ALL,Female,11,NaN,bone_marrow,NaN,NaN,NaN,DSMZ,file_9
2,ACH-000033,CVCL_1497,NCIH1819,NCIH1819_LUNG,Lung Cancer,"Non-Small Cell Lung Cancer (NSCLC), Adenocarci...",lung,NSCLC,Female,55,Metastasis,lymph_node,NaN,NaN,NaN,Academic lab,file_9
3,ACH-000043,CVCL_0993,HS895T,HS895T_FIBROBLAST,Non-Cancerous,Skin,fibroblast,fibroblast_skin,Female,48,Metastasis,fibroblast,2D: adherent,NaN,NaN,ATCC,file_9
4,ACH-000049,CVCL_WS59,HEKTE,HEKTE_KIDNEY,Non-Cancerous,NaN,kidney,NaN,NaN,NaN,NaN,kidney,NaN,NaN,NaN,Academic lab,file_9


Find orphan ACHs

In [124]:
# build PR -> ACH dictionary from File 8
profiles = pd.read_csv(DATA / 'nomenclature/8_DepMap_OmicsProfiles.csv')
pr_to_ach = dict(zip(profiles['ProfileID'], profiles['ModelID']))

ach_to_prs = profiles.groupby('ModelID')['ProfileID'].apply(list).to_dict()
print(f'PR->ACH mappings: {len(pr_to_ach)}')
print(f'ACHs with at least one PR: {len(ach_to_prs)}')

PR->ACH mappings: 3830
ACHs with at least one PR: 1822


In [125]:
orphans = set()
base_achs = set(dim['ach_id'])

# File 5 
f5 = pd.read_csv(DATA / 'gene properties/5_OmicsFusionFilteredSupplementary.csv',
                 usecols=['ModelID'])
orphans.update(set(f5['ModelID'].dropna()) - base_achs)

# File 14 
f14 = pd.read_csv(DATA / 'non gene expression/14_OmicsGlobalSignatures.csv',
                  usecols=['ModelID'])
orphans.update(set(f14['ModelID'].dropna()) - base_achs)

# File 2 
f2 = pd.read_csv(DATA / 'gene expression/2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv',
                 usecols=[0])
f2_achs = {pr_to_ach.get(pr) for pr in f2.iloc[:, 0] if pr in pr_to_ach}
f2_achs.discard(None)
orphans.update(f2_achs - base_achs)

# File 6 
f6 = pd.read_csv(DATA / 'gene properties/6_OmicsSomaticMutationsProfile.csv',
                 usecols=['ProfileID'])
f6_achs = {pr_to_ach.get(pr) for pr in set(f6['ProfileID'].dropna()) if pr in pr_to_ach}
f6_achs.discard(None)
orphans.update(f6_achs - base_achs)

print(f'orphan ACHs found: {len(orphans)}')

orphan ACHs found: 286


In [126]:
if orphans:
    orphan_rows = pd.DataFrame({'ach_id': sorted(orphans)})
    orphan_rows['metadata_source'] = 'fact_only'
    
    dim = pd.concat(
        [dim.reset_index(drop=True), orphan_rows.reset_index(drop=True)],
        ignore_index=True, sort=False
    )

print(f'after adding orphans: {len(dim)} rows')

after adding orphans: 2126 rows


Add HPA-only cell lines

In [127]:
hpa_desc = pd.read_csv(
    DATA / 'nomenclature/11_hpa_rna_celline_description.tsv',
    sep='\t'
)

existing_cvcls = set(dim['cvcl_id'].dropna())
hpa_cvcls = set(hpa_desc['Cellosaurus ID'].dropna())
hpa_only = hpa_cvcls - existing_cvcls

print(f'HPA cell lines total : {len(hpa_cvcls)}')
print(f'HPA-only (not in DepMap): {len(hpa_only)}')

HPA cell lines total : 1198
HPA-only (not in DepMap): 94


In [128]:
if hpa_only:
    hpa_rows = hpa_desc[hpa_desc['Cellosaurus ID'].isin(hpa_only)].copy()
    hpa_rows['ach_id'] = hpa_rows['Cellosaurus ID'].apply(lambda x: f'HPA_{x}')
    
    hpa_rows = hpa_rows.rename(columns={
        'Cellosaurus ID': 'cvcl_id',
        'Cell line': 'cell_line_name',
        'Disease': 'primary_disease',
        'Disease subtype': 'Subtype',
        'Primary/Metastasis': 'primary_or_metastasis',
        'Sample collection site': 'sample_collection_site',
    })
    hpa_rows['metadata_source'] = 'hpa_only'
    
    hpa_rows = hpa_rows[[
        'ach_id', 'cvcl_id', 'cell_line_name',
        'primary_disease', 'Subtype',
        'primary_or_metastasis', 'sample_collection_site',
        'metadata_source'
    ]].reset_index(drop=True)
    
    dim = pd.concat(
        [dim.reset_index(drop=True), hpa_rows],
        ignore_index=True, sort=False
    )

print(f'after adding HPA-only: {len(dim)} rows')

after adding HPA-only: 2220 rows


In [129]:
dim['profile_ids'] = dim['ach_id'].map(ach_to_prs)
dim['n_profiles'] = dim['profile_ids'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

dim = dim.sort_values('ach_id').reset_index(drop=True)

dim['metadata_source'].value_counts()

metadata_source
file_9       1840
fact_only     286
hpa_only       94
Name: count, dtype: int64

In [130]:
for col in ['cvcl_id', 'cell_line_name', 'primary_disease', 'lineage', 'growth_pattern']:
    n = dim[col].notna().sum()
    print(f'{col} {n} / {len(dim)}  ({n/len(dim):.1%})')

cvcl_id 1912 / 2220  (86.1%)
cell_line_name 1933 / 2220  (87.1%)
primary_disease 1934 / 2220  (87.1%)
lineage 1840 / 2220  (82.9%)
growth_pattern 1062 / 2220  (47.8%)


save

In [131]:
dim.to_parquet(PROCESSED / 'dim_cell_lines.parquet', index=False)
dim.to_csv(PROCESSED / 'dim_cell_lines.csv', index=False)
print(f'saved dim_cell_lines ({len(dim)} rows)')
dim.head()

saved dim_cell_lines (2220 rows)


,ach_id,cvcl_id,cell_line_name,ccle_name,primary_disease,Subtype,lineage,lineage_subtype,sex,age,primary_or_metastasis,sample_collection_site,growth_pattern,COSMICID,Sanger_Model_ID,source,metadata_source,profile_ids,n_profiles
0,ACH-000001,CVCL_0465,NIHOVCAR3,NIHOVCAR3_OVARY,Ovarian Cancer,"Adenocarcinoma, high grade serous",ovary,ovary_adenocarcinoma,Female,60,Metastasis,ascites,2D: adherent,905933.0,SIDM00105,ATCC,file_9,"[PR-99siqG, PR-sbeUkh, PR-vFBZ2h]",3
1,ACH-000002,CVCL_0002,HL60,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Leukemia,"Acute Myelogenous Leukemia (AML), M3 (Promyelo...",blood,AML,Female,36,Primary,haematopoietic_and_lymphoid_tissue,2D: suspension,905938.0,SIDM00829,ATCC,file_9,"[PR-qY1GC4, PR-yEldOx]",2
2,ACH-000003,CVCL_0025,CACO2,CACO2_LARGE_INTESTINE,Colon/Colorectal Cancer,Adenocarcinoma,colorectal,colorectal_adenocarcinoma,Male,72,NaN,Colon,NaN,NaN,SIDM00891,ATCC,file_9,[PR-jIbcm8],1
3,ACH-000004,CVCL_0001,HEL,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Leukemia,"Acute Myelogenous Leukemia (AML), M6 (Erythrol...",blood,AML,Male,30,NaN,haematopoietic_and_lymphoid_tissue,2D: suspension,907053.0,SIDM00594,DSMZ,file_9,"[PR-DGbP1m, PR-pEZmVx]",2
4,ACH-000005,CVCL_2481,HEL9217,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,Leukemia,"Acute Myelogenous Leukemia (AML), M6 (Erythrol...",blood,AML,Male,30,NaN,bone_marrow,2D: suspension,NaN,SIDM00593,ATCC,file_9,"[PR-7dTCvD, PR-dPW9OL, PR-y0OM3W, PR-ZMEDE0]",4


## dim_genes

Pull gene metadata from File 6 (mutations file has the richest gene info — symbol, ensembl, entrez, uniprot, biotype, chromosome).

Then add any UniProt IDs from the proteomics file that aren't already there.

In [132]:
mut_genes = pd.read_csv(
    DATA / 'gene properties/6_OmicsSomaticMutationsProfile.csv',
    usecols=['HugoSymbol', 'EnsemblGeneID', 'EntrezGeneID',
             'UniprotID', 'VepBiotype', 'Chrom']
)
print(f'loaded {len(mut_genes)} mutation rows for gene metadata')

loaded 1066869 mutation rows for gene metadata


In [133]:
mut_genes.head()

,Chrom,HugoSymbol,EnsemblGeneID,UniprotID,VepBiotype,EntrezGeneID
0,chr1,FAM87B,ENSG00000177757,NaN,lncRNA,400728.0
1,chr1,LINC01128,ENSG00000228794,NaN,lncRNA,643837.0
2,chr1,SAMD11,ENSG00000187634,NaN,protein_coding,148398.0
3,chr1,SAMD11,ENSG00000187634,NaN,protein_coding,148398.0
4,chr1,SAMD11,ENSG00000187634,NaN,protein_coding,148398.0


In [134]:
mut_genes['ensembl_id_clean'] = (
    mut_genes['EnsemblGeneID'].astype(str).str.split('.').str[0]
)

mut_genes = mut_genes[
    mut_genes['ensembl_id_clean'].notna() &
    (mut_genes['ensembl_id_clean'] != 'nan')
]

mut_genes['ensembl_id_clean'].head(5)

0    ENSG00000177757
1    ENSG00000228794
2    ENSG00000187634
3    ENSG00000187634
4    ENSG00000187634
Name: ensembl_id_clean, dtype: object

In [135]:
# group by ensembl_id so each gene only appears once
gene_records = (
    mut_genes.groupby('ensembl_id_clean')
    .agg({
        'HugoSymbol': 'first',
        'EntrezGeneID': 'first',
        'UniprotID': 'first',
        'VepBiotype': 'first',
        'Chrom': 'first',
    })
    .reset_index()
    .rename(columns={
        'ensembl_id_clean': 'ensembl_id',
        'HugoSymbol': 'hugo_symbol',
        'EntrezGeneID': 'entrez_id',
        'UniprotID': 'uniprot_id',
        'VepBiotype': 'gene_type',
        'Chrom': 'chromosome',
    })
)

print(f'unique genes from mutations: {len(gene_records)}')
gene_records.head()

unique genes from mutations: 19798


,ensembl_id,hugo_symbol,entrez_id,uniprot_id,gene_type,chromosome
0,ENSG00000000003,TSPAN6,7105.0,None,protein_coding,chrX
1,ENSG00000000005,TNMD,64102.0,Q9H2S6-1,protein_coding,chrX
2,ENSG00000000419,DPM1,8813.0,None,protein_coding,chr20
3,ENSG00000000457,SCYL3,57147.0,Q8IZE3-2,protein_coding,chr1
4,ENSG00000000460,C1orf112,55732.0,Q9NSG2-1,protein_coding_CDS_not_defined,chr1


Add protein-only genes

In [136]:
import re

protein_header = pd.read_csv(
    DATA / 'gene expression/4_Harmonized_MS_CCLE_Gygi_subsetted.csv',
    nrows=0
)
protein_cols = list(protein_header.columns)[1:]

In [137]:
protein_genes = []
for col in protein_cols:
    m = re.match(r'^(\w+)\s*\(([^)]+)\)$', col)
    if m:
        protein_genes.append({
            'uniprot_id': m.group(1),
            'hugo_symbol': m.group(2),
        })

protein_genes_df = pd.DataFrame(protein_genes)
print(f'genes from proteomics: {len(protein_genes_df)}')
protein_genes_df.head()

genes from proteomics: 10899


,uniprot_id,hugo_symbol
0,A0AV96,RBM47
1,A0AVF1,IFT56
2,A0AVG3,TSNARE1
3,A0AVI4,TMEM129
4,A0AVK6,E2F8


In [ ]:
existing_symbols = set(gene_records['hugo_symbol'].dropna())

protein_only_uniprots = (set(protein_genes_df['hugo_symbol']) - existing_symbols)

print(f'protein-only UniProt IDs: {len(protein_only_uniprots)}')

if protein_only_uniprots:
    extra = protein_genes_df[
        protein_genes_df['hugo_symbol'].isin(protein_only_uniprots)
    ].copy()
    
    extra['ensembl_id'] = 'UPROT_' + extra['uniprot_id']
    extra['gene_type'] = 'protein_coding'
    
    gene_records = pd.concat([gene_records, extra], ignore_index=True)

gene_records = gene_records.sort_values('ensembl_id').reset_index(drop=True)
print(f'final dim_genes: {len(gene_records)} rows')

protein-only UniProt IDs: 334
final dim_genes: 20135 rows


In [139]:
# coverage check
for col in ['hugo_symbol', 'entrez_id', 'uniprot_id', 'gene_type']:
    n = gene_records[col].notna().sum()
    print(f'{col} {n} / {len(gene_records)}  ({n/len(gene_records):.1%})')

hugo_symbol 20135 / 20135  (100.0%)
entrez_id 18602 / 20135  (92.4%)
uniprot_id 9905 / 20135  (49.2%)
gene_type 20135 / 20135  (100.0%)


save

In [140]:
gene_records.to_parquet(PROCESSED / 'dim_genes.parquet', index=False)
gene_records.to_csv(PROCESSED / 'dim_genes.csv', index=False)
print(f'saved dim_genes ({len(gene_records)} rows)')
gene_records.head()

saved dim_genes (20135 rows)


,ensembl_id,hugo_symbol,entrez_id,uniprot_id,gene_type,chromosome
0,ENSG00000000003,TSPAN6,7105.0,None,protein_coding,chrX
1,ENSG00000000005,TNMD,64102.0,Q9H2S6-1,protein_coding,chrX
2,ENSG00000000419,DPM1,8813.0,None,protein_coding,chr20
3,ENSG00000000457,SCYL3,57147.0,Q8IZE3-2,protein_coding,chr1
4,ENSG00000000460,C1orf112,55732.0,Q9NSG2-1,protein_coding_CDS_not_defined,chr1
